<a href="https://colab.research.google.com/github/anurag004-coder/Streamlit_Projects/blob/main/Context_Aware_Sentiment_Analyazer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# Cell 1: Install Hugging Face ecosystems and text utilities
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [13]:
import pandas as pd


In [17]:
# Load the CSV without headers and assign proper column names
df = pd.read_csv("/content/financial_phrasebank.csv",
                 encoding='ISO-8859-1',
                 header=None,
                 names=['sentiment', 'text'])

# Display the first few rows to confirm it looks correct
df.head()

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [20]:
# Cell 3: Baseline Classifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Fix: Access columns directly from the DataFrame instead of iterating over it
texts = df['text'].values
labels = df['sentiment'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Vectorize text using TF-IDF (Term Frequency-Inverse Document Frequency)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train a baseline Logistic Regression model
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

# Evaluate
preds = baseline_model.predict(X_test_tfidf)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

    negative       0.92      0.49      0.64       110
     neutral       0.76      0.94      0.84       571
    positive       0.79      0.55      0.65       289

    accuracy                           0.77       970
   macro avg       0.82      0.66      0.71       970
weighted avg       0.78      0.77      0.76       970



In [23]:
# Cell 4: Tokenization & Transformer Setup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import torch

# Map string labels to integers for the model
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df['label'] = df['sentiment'].map(label_map)

# Convert pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[['text', 'label']])

# Using DistilBERT for faster training
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Note: Using 'text' as the key to match our DataFrame column name
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Split and tokenize
dataset_dict = hf_dataset.train_test_split(test_size=0.2, seed=42)
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Correctly load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
from transformers import TrainingArguments, Trainer

# Cell 5: Fine-tuning on GPU
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # Fixed: evaluation_strategy is now eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True # Speeds up training on the T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Run the training loop
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.434860
2,No log,0.417424
3,0.432226,0.450391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=729, training_loss=0.35838351884169506, metrics={'train_runtime': 166.4844, 'train_samples_per_second': 69.844, 'train_steps_per_second': 4.379, 'total_flos': 1540358381187072.0, 'train_loss': 0.35838351884169506, 'epoch': 3.0})

In [29]:
import numpy as np
from evaluate import load

# Load accuracy metric
metric = load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Update the trainer with the metric function and evaluate
trainer.compute_metrics = compute_metrics
eval_results = trainer.evaluate()

print("Final Evaluation Results:")
display(eval_results)

Final Evaluation Results:


{'eval_loss': 0.4174281358718872,
 'eval_accuracy': 0.8329896907216495,
 'eval_runtime': 3.8081,
 'eval_samples_per_second': 254.721,
 'eval_steps_per_second': 16.019,
 'epoch': 3.0}

In [30]:
# Cell 6: Install UI dependencies
!pip install -q gradio

In [31]:
# Cell 7: Launch an interactive dashboard inside your Colab notebook
import gradio as gr
from transformers import pipeline

# Build a fast inference pipeline using your fine-tuned model
nlp_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

# Map labels back to human-readable strings
label_map = {"LABEL_0": "Negative 📉", "LABEL_1": "Neutral ➡️", "LABEL_2": "Positive 📈"}

def analyze_context_sentiment(text):
    prediction = nlp_pipeline(text)[0]
    readable_label = label_map.get(prediction['label'], prediction['label'])
    return f"Prediction: {readable_label} \nConfidence Score: {prediction['score']:.2f}"

# Create a sleek web UI layout
demo = gr.Interface(
    fn=analyze_context_sentiment,
    inputs=gr.Textbox(placeholder="Enter a financial sentence... (e.g., 'Operating profit rose by 12%')"),
    outputs="text",
    title="Domain-Specific Sentiment Analyzer",
    description="An AI model specialized in parsing contextual business and financial sentiment accurately."
)

# Launch directly in your browser tab via a temporary public link
demo.launch(share=True)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
I0604 13:00:01.143921 958 _client.py:1025] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
I0604 13:00:01.344088 958 _client.py:1025] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
I0604 13:00:01.524421 958 _client.py:1025] HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
I0604 13:00:01.573113 958 _client.py:1025] HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


INFO:httpx:HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
I0604 13:00:01.800508 958 _client.py:1025] HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
I0604 13:00:01.922071 958 _client.py:1025] HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://33385ec757a9a7552c.gradio.live "HTTP/1.1 200 OK"
I0604 13:00:02.468701 958 _client.py:1025] HTTP Request: HEAD https://33385ec757a9a7552c.gradio.live "HTTP/1.1 200 OK"


* Running on public URL: https://33385ec757a9a7552c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
